# AlphaClash: train the unit detector (YOLOv8)

Runtime > Change runtime type > **T4 GPU**. Upload `alphaclash_yolo.zip` to the root of your Google Drive first (`MyDrive/alphaclash_yolo.zip`). Weights are saved back to `MyDrive/alphaclash_runs/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install ultralytics==8.4.183
!cp /content/drive/MyDrive/alphaclash_yolo.zip /content/ && cd /content && unzip -q -o alphaclash_yolo.zip && ls alphaclash_yolo

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8s.pt')
# optimizer must be explicit: 'auto' picks AdamW at lr 0.002*5/(4+nc), only 6e-5 for
# our 153 classes, and run 1 (yolov8s_katacr) plateaued at mAP50 0.38 by epoch 30.
model.train(
    data='/content/alphaclash_yolo/dataset.yaml',
    imgsz=640, epochs=80, batch=32, device=0, workers=4,
    optimizer='SGD', lr0=0.01, momentum=0.937, cos_lr=True,
    fliplr=0.5, mosaic=1.0, close_mosaic=10,
    project='/content/drive/MyDrive/alphaclash_runs', name='yolov8s_katacr_sgd', exist_ok=True,
)

In [ ]:
metrics = model.val()
print('mAP50', metrics.box.map50, 'mAP50-95', metrics.box.map)
# weakest classes, to decide where synthetic examples are worth adding
worst = sorted(zip(metrics.box.ap50, (model.names[i] for i in metrics.box.ap_class_index)))[:25]
for ap, name in worst: print(f'{ap:.2f}  {name}')